# HybridMSLATNet on APTOS (clean pipeline)

**Run the cells top to bottom, once.** Needs a **GPU runtime** (it also runs on CPU, but very slowly).

Order: setup (1-3) → imports & config (4-5) → transforms (6) → data + duplicate cleaning (7) → quantum circuit & model (8-10) → loss (11) → model/optimizer (12) → training (13) → test evaluation (14) → curves (15).

**Smoke test first:** in the config cell set `"epochs": 3` and `"tag": "smoke"`, run everything, check there are no errors, then set `"epochs": 50` and `"tag": ""` and run again (the tag keeps the checkpoints of the two runs separate).

The Grad-CAM / explainability cells are intentionally **not** included. We will rewrite them after training.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
IDRID_ZIP = "/content/drive/MyDrive/IDRID.zip"     # change to the real path of your IDRiD zip in Drive

if not os.path.exists("/content/idrid/IDRID"):
    !unzip -q "{IDRID_ZIP}" -d /content/idrid
print(os.listdir("/content/idrid"))

['IDRID']


In [ ]:
!pip install timm pennylane pennylane-lightning -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 17.4 MB/s eta 0:00:00


## 2. Imports and configuration

In [ ]:
import os, random, hashlib
import numpy as np, pandas as pd, torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import timm
import pennylane as qml
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score, roc_auc_score, classification_report, confusion_matrix)

In [ ]:
BASE_PATH = "/content/drive/MyDrive/your folder"
os.makedirs(BASE_PATH + "/checkpoints", exist_ok=True)

SEED = 42          # change to 43, 44 for extra seeds

config = {
    "epochs": 50,
    "batch_size": 32,
    "num_classes": 5,
    "lr_backbone": 1e-4,
    "lr_other": 1e-4,
    "lr_alpha": 1e-2,        # raised so alpha can really move (APTOS gives only ~88 steps per epoch)
    "weight_decay": 0.01,

    # ---- experiment switches: the defaults below are the BASELINE model ----
    "quantum_mode": "circuit",   # "circuit" | "skip" (no circuit, same 8 numbers) | "mlp" (matched classical layer) | "frozen" (random untrained circuit)
    "fixed_alpha": None,         # None = learnable. Or a number (0.0, 0.27, 0.5, 1.0) to fix the mixing weight
    "use_pos_emb": False,        # True = add learned position/scale embeddings to the tokens
    "n_trans_layers": 4,         # Transformer depth (paper text says 2 - decide using validation results)

    "tag": "v3",                   # e.g. "smoke" for the quick test run
}

RUN = (f"{config['tag']}_" if config["tag"] else "") + \
      f"{config['quantum_mode']}_a{config['fixed_alpha']}_pe{int(config['use_pos_emb'])}_L{config['n_trans_layers']}_s{SEED}"
print("RUN name:", RUN)

RUN name: v3_circuit_aNone_pe0_L4_s42


## 3. Augmentation
Colour-neutral (no hue shift, no CLAHE, no random erasing); the same preprocessing for every split.

In [ ]:
train_tf = A.Compose([
    A.Resize(256, 256),
    A.RandomResizedCrop(size=(224, 224), scale=(0.85, 1.0)),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    A.Rotate(limit=15),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_tf = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

## 4. Data: read APTOS, remove exact duplicates, build loaders
The first run hashes every image (about 1.5 minutes) and saves `aptos_clean_split.csv`; later runs just load that file.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ---------- read the three label files ----------
ROOT = "/content/APTOS"
tr = pd.read_csv(f"{ROOT}/train_1.csv"); tr["dir"] = f"{ROOT}/train_images/train_images"
va = pd.read_csv(f"{ROOT}/valid.csv");   va["dir"] = f"{ROOT}/val_images/val_images"
te = pd.read_csv(f"{ROOT}/test.csv");    te["dir"] = f"{ROOT}/test_images/test_images"

print("rows  train/valid/test:", len(tr), len(va), len(te))
print("ids shared train/valid:", len(set(tr.id_code) & set(va.id_code)),
      "| train/test:", len(set(tr.id_code) & set(te.id_code)),
      "| valid/test:", len(set(va.id_code) & set(te.id_code)))

# ---------- remove exact duplicates (cached) ----------
CLEAN_CSV = BASE_PATH + "/aptos_clean_split.csv"
if os.path.exists(CLEAN_CSV):
    clean = pd.read_csv(CLEAN_CSV)
    print("Loaded cleaned split from", CLEAN_CSV)
else:
    pool = pd.concat([tr, va, te]).reset_index(drop=True)
    pool["split"] = ["train"] * len(tr) + ["val"] * len(va) + ["test"] * len(te)
    path_of = lambda r: f"{r['dir']}/{r['id_code']}.png"
    pool["md5"] = [hashlib.md5(open(path_of(r), "rb").read()).hexdigest()
                   for _, r in tqdm(pool.iterrows(), total=len(pool))]

    order = {"train": 0, "val": 1, "test": 2}
    keep, removed_dup, removed_conflict = [], 0, 0
    for h, grp in pool.groupby("md5"):
        if len(grp) == 1:
            keep.append(grp.index[0]); continue
        if grp["diagnosis"].nunique() > 1:          # same photo with different labels -> drop all copies
            removed_conflict += len(grp); continue
        keep.append(grp.assign(o=grp["split"].map(order)).sort_values("o").index[0])   # earliest split wins
        removed_dup += len(grp) - 1

    clean = pool.loc[sorted(keep)]
    print(f"before: {len(pool)} | duplicates removed: {removed_dup} | removed (conflicting labels): {removed_conflict} | after: {len(clean)}")
    clean.to_csv(CLEAN_CSV, index=False)

import cv2
from concurrent.futures import ThreadPoolExecutor
CACHE = "/content/aptos_256"; os.makedirs(CACHE, exist_ok=True)

def cache_one(a):
    src, dst = a
    if not os.path.exists(dst):
        im = cv2.imread(src)
        cv2.imwrite(dst, cv2.resize(im, (256, 256), interpolation=cv2.INTER_AREA))

jobs = [(f"{r['dir']}/{r['id_code']}.png", f"{CACHE}/{r['id_code']}.png") for _, r in clean.iterrows()]
with ThreadPoolExecutor(4) as ex:
    list(tqdm(ex.map(cache_one, jobs), total=len(jobs)))
clean["dir"] = CACHE          # loaders now read the small images (in memory only; the saved CSV is unchanged)
train_df = clean[clean["split"] == "train"]
val_df   = clean[clean["split"] == "val"]
test_df  = clean[clean["split"] == "test"]
for nm, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{nm:5s} {len(d):5d}", np.bincount(d["diagnosis"], minlength=5))

# ---------- dataset + loaders ----------
class APTOSDF(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.t = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = np.array(Image.open(os.path.join(r["dir"], r["id_code"] + ".png")).convert("RGB"))
        return self.t(image=img)["image"], int(r["diagnosis"])

def seed_worker(worker_id):
    s = torch.initial_seed() % 2**32
    np.random.seed(s); random.seed(s)

g = torch.Generator(); g.manual_seed(SEED)
bs = config["batch_size"]
train_loader = DataLoader(APTOSDF(train_df, train_tf), batch_size=bs, shuffle=True, num_workers=2,
                          pin_memory=True, persistent_workers=True,generator=g, worker_init_fn=seed_worker, drop_last=True)
val_loader   = DataLoader(APTOSDF(val_df,  val_tf), batch_size=bs, shuffle=False, persistent_workers=True,num_workers=2, pin_memory=True)
test_loader  = DataLoader(APTOSDF(test_df, val_tf), batch_size=bs, shuffle=False, persistent_workers=True,num_workers=2, pin_memory=True)

## 5. Quantum circuit (8 qubits, 3 layers, data re-uploading, ring CNOT)

In [ ]:
n_qubits = 8
n_layers = 3

dev = qml.device("default.qubit", wires=n_qubits)

def quantum_circuit(inputs, weights):
    for i in range(n_qubits):                       # initial U3 encoding
        qml.U3(inputs[..., i], inputs[..., i], inputs[..., i], wires=i)
    for l in range(n_layers):
        for i in range(n_qubits):                   # data re-uploading
            qml.RY(inputs[..., i], wires=i)
        for i in range(n_qubits):                   # trainable single-qubit rotations (72 parameters in total)
            qml.U3(weights[l][i][0], weights[l][i][1], weights[l][i][2], wires=i)
        for i in range(n_qubits):                   # ring entanglement
            qml.CNOT(wires=[i, (i + 1) % n_qubits])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

weight_shapes = {"weights": (n_layers, n_qubits, 3)}
qnode = qml.QNode(quantum_circuit, dev, interface="torch", diff_method="backprop")

In [ ]:
import time
w = {"weights": (n_layers, n_qubits, 3)}
q_old = qml.qnn.TorchLayer(qml.QNode(quantum_circuit, qml.device("lightning.qubit", wires=8), interface="torch", diff_method="adjoint"), w)
q_new = qml.qnn.TorchLayer(qml.QNode(quantum_circuit, qml.device("default.qubit", wires=8), interface="torch", diff_method="backprop"), w)
q_new.load_state_dict(q_old.state_dict())

xin = (torch.rand(32, 8) * 2 - 1) * np.pi
print("max output difference:", (q_old(xin) - q_new(xin)).abs().max().item())      # expect ~1e-6 or smaller

# gradients should also agree
xa = xin.clone().requires_grad_(True); xb = xin.clone().requires_grad_(True)
q_old(xa).sum().backward(); q_new(xb).sum().backward()
print("max input-gradient difference:", (xa.grad - xb.grad).abs().max().item())

t = time.time()
for _ in range(3): q_new(xin.clone().requires_grad_(True)).sum().backward()
print(f"new layer on CPU: {(time.time()-t)/3:.2f} s per batch")

try:
    q_gpu = q_new.to(device); xg = xin.to(device).requires_grad_(True)
    q_gpu(xg).sum().backward(); torch.cuda.synchronize(); t = time.time()
    for _ in range(3): q_gpu(xg).sum().backward()
    torch.cuda.synchronize(); print(f"new layer on GPU: {(time.time()-t)/3:.2f} s per batch")
except Exception as e:
    print("GPU test failed:", type(e).__name__, str(e)[:150])

max output difference: 2.384185791015625e-07
max input-gradient difference: 1.3113021850585938e-06
new layer on CPU: 0.25 s per batch
GPU test failed: NameError name 'device' is not defined


In [ ]:
class QuantumBranch(nn.Module):
    """256 -> attention gate -> reducer -> 8 angles -> (circuit | control) -> 8 features"""
    def __init__(self, feature_dim=256, mode="circuit"):
        super().__init__()
        self.mode = mode

        self.attn = nn.Sequential(
            nn.Linear(feature_dim, 64), nn.ReLU(), nn.Linear(64, feature_dim)
        )
        self.reducer = nn.Sequential(
            nn.Linear(feature_dim, 256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, 128),         nn.GELU(), nn.Dropout(0.2),
            nn.Linear(128, 64),          nn.GELU(), nn.Dropout(0.2),
            nn.Linear(64, n_qubits)
        )

        if mode in ("circuit", "frozen"):
            self.quantum = qml.qnn.TorchLayer(qnode, weight_shapes)   # a fresh circuit for every model
            if mode == "frozen":                                      # random circuit that is never trained
                for p in self.quantum.parameters():
                    p.requires_grad_(False)
        elif mode == "mlp":                                           # matched classical layer: 8*8 + 8 = 72 parameters
            self.quantum = nn.Sequential(nn.Linear(n_qubits, n_qubits), nn.Tanh())
        elif mode == "skip":                                          # no circuit at all
            self.quantum = None
        else:
            raise ValueError(f"unknown quantum_mode: {mode}")

    def forward(self, x):
        a = torch.softmax(self.attn(x), dim=-1)
        x = x * a
        x = self.reducer(x)
        x = F.dropout(x, p=0.2, training=self.training)
        x = torch.tanh(x) * np.pi                       # 8 angles in (-pi, pi)

        if self.mode in ("circuit", "frozen"):
            q_out = self.quantum(x.float())
        elif self.mode == "mlp":
            q_out = self.quantum(x.float() / np.pi)
        else:                                            # "skip": same 8 numbers, scaled to (-1, 1) like the circuit output
            q_out = x.float() / np.pi
        return q_out, x

In [ ]:
class HybridMSLATNet(nn.Module):
    def __init__(self, num_classes=5, embed_dim=256, n_trans_layers=4, use_pos_emb=False,
                 quantum_mode="circuit", fixed_alpha=None):
        super().__init__()

        self.backbone = timm.create_model("efficientnet_b2", pretrained=True, features_only=True)
        feat_dims = [f["num_chs"] for f in self.backbone.feature_info[-3:]]

        self.proj = nn.ModuleList([nn.Linear(feat_dims[i], embed_dim) for i in range(3)])

        # optional learned position / scale embeddings (one vector per token of the concatenated sequence)
        self.pos_emb = None
        if use_pos_emb:
            was_training = self.backbone.training
            self.backbone.eval()
            with torch.no_grad():
                fs = self.backbone(torch.zeros(1, 3, 224, 224))[-3:]
            n_tokens = sum(f.shape[2] * f.shape[3] for f in fs)
            self.backbone.train(was_training)
            self.pos_emb = nn.Parameter(torch.zeros(1, n_tokens, embed_dim))
            nn.init.trunc_normal_(self.pos_emb, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=8, dim_feedforward=embed_dim * 4,
            dropout=0.5, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=n_trans_layers, enable_nested_tensor=False)

        self.pool = nn.Linear(embed_dim, 1)

        self.classical_head = nn.Sequential(
            nn.Linear(embed_dim, 128), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(128, 64),        nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(64, num_classes)
        )

        self.quantum_branch = QuantumBranch(embed_dim, mode=quantum_mode)

        self.fusion_head = nn.Sequential(
            nn.Linear(embed_dim + n_qubits, 256), nn.GELU(), nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
        self.quantum_head = nn.Sequential(
            nn.Linear(n_qubits, 32), nn.GELU(), nn.Linear(32, num_classes)
        )

        # mixing weight between the classical and the fused prediction
        self.learn_alpha = fixed_alpha is None
        if self.learn_alpha:
            self.alpha = nn.Parameter(torch.tensor(-1.0))               # sigmoid(-1) = 0.269 at the start
        else:
            self.register_buffer("alpha", torch.tensor(float(fixed_alpha)))

    def get_alpha(self):
        return torch.sigmoid(self.alpha) if self.learn_alpha else self.alpha

    def forward(self, x):
        feats = self.backbone(x)[-3:]

        tokens = []
        for i, f in enumerate(feats):
            t = f.flatten(2).transpose(1, 2)          # (B, H*W, C): every spatial position is one token
            tokens.append(self.proj[i](t))
        tokens = torch.cat(tokens, dim=1)
        if self.pos_emb is not None:
            tokens = tokens + self.pos_emb
        tokens = self.transformer(tokens)

        attn = torch.softmax(self.pool(tokens), dim=1)
        pooled = (tokens * attn).sum(dim=1)            # (B, 256)

        classical_logits = self.classical_head(pooled)

        quantum_features, q_input = self.quantum_branch(pooled)
        fusion_logits  = self.fusion_head(torch.cat([pooled, quantum_features], dim=1))
        quantum_logits = self.quantum_head(quantum_features)

        alpha = self.get_alpha()
        out = (1 - alpha) * classical_logits + alpha * fusion_logits

        return out, classical_logits, quantum_logits, pooled, attn, q_input

In [ ]:
   for pre in ["v3_circuit_aNone_pe0_L4", "skip50_skip_aNone_pe0_L4"]:
       for s in (42, 43, 44):
           f = f"{BASE_PATH}/checkpoints/best_{pre}_s{s}.pth"
           print(os.path.basename(f), "OK" if os.path.exists(f) else "MISSING")

best_v3_circuit_aNone_pe0_L4_s42.pth OK
best_v3_circuit_aNone_pe0_L4_s43.pth OK
best_v3_circuit_aNone_pe0_L4_s44.pth OK
best_skip50_skip_aNone_pe0_L4_s42.pth OK
best_skip50_skip_aNone_pe0_L4_s43.pth OK
best_skip50_skip_aNone_pe0_L4_s44.pth OK


In [ ]:
# ===== External test on IDRiD (all 516 graded images). Six saved checkpoints, no retraining =====
import os, cv2, numpy as np, pandas as pd, torch
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, confusion_matrix

IDRID_ROOT = "/content/idrid/IDRID"
img_dirs  = {"train": f"{IDRID_ROOT}/1. Original Images/a. Training Set",
             "test":  f"{IDRID_ROOT}/1. Original Images/b. Testing Set"}
lab_files = {"train": f"{IDRID_ROOT}/2. Groundtruths/a. IDRiD_Disease Grading_Training Labels.csv",
             "test":  f"{IDRID_ROOT}/2. Groundtruths/b. IDRiD_Disease Grading_Testing Labels.csv"}

# ---- 1. build the table (image names repeat across the two subsets, so keep them apart) ----
rows = []
for sub in ("train", "test"):
    lab = pd.read_csv(lab_files[sub]).iloc[:, :2].dropna()
    lab.columns = ["name", "grade"]
    files = {os.path.splitext(f)[0]: os.path.join(img_dirs[sub], f) for f in os.listdir(img_dirs[sub])
             if f.lower().endswith((".jpg", ".jpeg", ".png", ".tif", ".tiff"))}
    for _, r in lab.iterrows():
        n = str(r["name"]).strip()
        if n in files: rows.append(dict(subset=sub, name=n, path=files[n], diagnosis=int(r["grade"])))
        else: print("no image found for", sub, n)
ext = pd.DataFrame(rows)
print(len(ext), "images |", ext.subset.value_counts().to_dict())
print("grade counts (0-4):", np.bincount(ext.diagnosis, minlength=5))

# ---- 2. same preprocessing as APTOS: 256x256 (area interpolation), then 224 + ImageNet normalization ----
CACHE = "/content/idrid_256"; os.makedirs(CACHE, exist_ok=True)
ext["cache"] = [f"{CACHE}/{s}_{n}.png" for s, n in zip(ext.subset, ext.name)]
def cache_one(a):
    src, dst = a
    if not os.path.exists(dst):
        cv2.imwrite(dst, cv2.resize(cv2.imread(src), (256, 256), interpolation=cv2.INTER_AREA))
with ThreadPoolExecutor(4) as ex:
    list(tqdm(ex.map(cache_one, zip(ext.path, ext.cache)), total=len(ext)))

val_tf = A.Compose([A.Resize(224, 224), A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), ToTensorV2()])
class ExtDS(Dataset):
    def __init__(self, df): self.p, self.y = df.cache.tolist(), df.diagnosis.tolist()
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return val_tf(image=cv2.cvtColor(cv2.imread(self.p[i]), cv2.COLOR_BGR2RGB))["image"], self.y[i]
ext_loader = DataLoader(ExtDS(ext), batch_size=32, shuffle=False, num_workers=2)
Y = ext.diagnosis.values

# ---- 3. evaluate every checkpoint ----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu"); use_amp = device.type == "cuda"
runs = [("circuit", f"v3_circuit_aNone_pe0_L4_s{s}") for s in (42, 43, 44)] + \
       [("skip", f"skip50_skip_aNone_pe0_L4_s{s}") for s in (42, 43, 44)]
L = [0, 1, 2, 3, 4]

@torch.no_grad()
def predict(m):
    m.eval(); P = []
    for imgs, _ in ext_loader:
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = m(imgs.to(device))[0]
        P.append(torch.softmax(out.float(), 1).cpu().numpy())
    P = np.concatenate(P).astype(np.float64)
    return P / P.sum(1, keepdims=True)

def summarize(y, P):
    pr = P.argmax(1); cm = confusion_matrix(y, pr, labels=L)
    return dict(acc=accuracy_score(y, pr), f1=f1_score(y, pr, average="macro", labels=L, zero_division=0),
                qwk=cohen_kappa_score(y, pr, weights="quadratic", labels=L),
                auc=roc_auc_score(y, P, multi_class="ovr", labels=L),
                sens=cm[2:, 2:].sum() / cm[2:].sum(), spec=cm[:2, :2].sum() / cm[:2].sum()), cm

allP, out_rows = {}, []
for mode, RUNN in runs:
    ck = torch.load(f"{BASE_PATH}/checkpoints/best_{RUNN}.pth", map_location=device, weights_only=False)
    m = HybridMSLATNet(n_trans_layers=4, use_pos_emb=False, quantum_mode=mode, fixed_alpha=None).to(device)
    m.load_state_dict(ck["model"])
    P = predict(m); allP[RUNN] = P
    np.savez(f"{BASE_PATH}/ext_idrid_{RUNN}.npz", P=P, Y=Y)
    s, _ = summarize(Y, P)
    out_rows.append(dict(run=RUNN, family=mode, dataset="idrid_516", n=len(Y), **s))
    print(RUNN, {k: round(float(v), 4) for k, v in s.items()})
    del m
    if device.type == "cuda": torch.cuda.empty_cache()

res = pd.DataFrame(out_rows)
print("\nmean / std by family:")
print(res.groupby("family")[["acc", "f1", "qwk", "auc", "sens", "spec"]].agg(["mean", "std"]).round(3).to_string())
for fam in ("circuit", "skip"):
    s, cm = summarize(Y, np.mean([allP[r] for (md, r) in runs if md == fam], 0))
    print(f"\n{fam} 3-seed ensemble:", {k: round(float(v), 4) for k, v in s.items()})
    print(cm)
res.to_csv(f"{BASE_PATH}/results_external_idrid.csv", index=False)

516 images | {'train': 413, 'test': 103}
grade counts (0-4): [168  25 168  93  62]


100%|██████████| 516/516 [00:51<00:00, 10.03it/s]


model.safetensors: reconstructing file:   0%|          |  0.00B / 36.8MB            

model.safetensors: downloading bytes:           |  0.00B            

v3_circuit_aNone_pe0_L4_s42 {'acc': 0.4593, 'f1': 0.4156, 'qwk': 0.6832, 'auc': 0.78, 'sens': 0.8235, 'spec': 0.9119}


v3_circuit_aNone_pe0_L4_s43 {'acc': 0.4942, 'f1': 0.4313, 'qwk': 0.6882, 'auc': 0.7846, 'sens': 0.8545, 'spec': 0.9016}


v3_circuit_aNone_pe0_L4_s44 {'acc': 0.4884, 'f1': 0.4332, 'qwk': 0.5691, 'auc': 0.8222, 'sens': 0.9288, 'spec': 0.6425}


skip50_skip_aNone_pe0_L4_s42 {'acc': 0.4806, 'f1': 0.4176, 'qwk': 0.6271, 'auc': 0.8045, 'sens': 0.904, 'spec': 0.7254}


skip50_skip_aNone_pe0_L4_s43 {'acc': 0.4651, 'f1': 0.4099, 'qwk': 0.6092, 'auc': 0.8175, 'sens': 0.9133, 'spec': 0.6891}


skip50_skip_aNone_pe0_L4_s44 {'acc': 0.4554, 'f1': 0.3546, 'qwk': 0.6114, 'auc': 0.7689, 'sens': 0.9226, 'spec': 0.7358}

mean / std by family:
           acc            f1           qwk           auc          sens          spec       
          mean    std   mean    std   mean    std   mean    std   mean    std   mean    std
family                                                                                     
circuit  0.481  0.019  0.427  0.010  0.647  0.067  0.796  0.023  0.869  0.054  0.819  0.153
skip     0.467  0.013  0.394  0.034  0.616  0.010  0.797  0.025  0.913  0.009  0.717  0.024

circuit 3-seed ensemble: {'acc': 0.4942, 'f1': 0.4422, 'qwk': 0.6999, 'auc': 0.829, 'sens': 0.8638, 'spec': 0.8912}
[[ 74  75  14   1   4]
 [  5  18   1   0   1]
 [  5  34 116   9   4]
 [  0   3  55  21  14]
 [  0   2  22  12  26]]

skip 3-seed ensemble: {'acc': 0.4942, 'f1': 0.4273, 'qwk': 0.6564, 'auc': 0.8187, 'sens': 0.9195, 'spec': 0.7513}
[[ 65  61  38   0   4]
 [  3  16   6   0   0]
 [

In [ ]:
import zipfile
if not os.path.exists("/content/APTOS"):
    zipfile.ZipFile("/content/drive/MyDrive/archive.zip").extractall("/content/APTOS")
a = pd.read_csv(BASE_PATH + "/aptos_clean_split.csv"); a = a[a["split"] == "test"].reset_index(drop=True)
a["path"] = [f"/content/APTOS/test_images/test_images/{i}.png" for i in a.id_code]
os.makedirs("/content/aptos_chk", exist_ok=True)
a["cache"] = [f"/content/aptos_chk/{i}.png" for i in a.id_code]
for s_, d_ in zip(a.path, a.cache): cache_one((s_, d_))
ext_loader = DataLoader(ExtDS(a), batch_size=32, shuffle=False, num_workers=2)   # predict() reads this global
ck = torch.load(f"{BASE_PATH}/checkpoints/best_v3_circuit_aNone_pe0_L4_s42.pth", map_location=device, weights_only=False)
m = HybridMSLATNet(n_trans_layers=4, use_pos_emb=False, quantum_mode="circuit", fixed_alpha=None).to(device)
m.load_state_dict(ck["model"])
print({k: round(float(v), 4) for k, v in summarize(a.diagnosis.values, predict(m))[0].items()})   # expect acc 0.8407, qwk 0.8759

{'acc': 0.8407, 'f1': 0.6017, 'qwk': 0.8759, 'auc': 0.9313, 'sens': 0.9386, 'spec': 0.9422}


In [ ]:
def crop_retina(img, thr=12, frac=0.02):
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY); m = g > thr
    rows = np.where(m.mean(1) > frac)[0]; cols = np.where(m.mean(0) > frac)[0]
    if len(rows) == 0 or len(cols) == 0: return img
    c = img[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]
    h, w = c.shape[:2]; s = max(h, w)
    out = np.zeros((s, s, 3), dtype=c.dtype)
    out[(s - h) // 2:(s - h) // 2 + h, (s - w) // 2:(s - w) // 2 + w] = c
    return out

def cache_crop(a):
    src, dst = a
    if not os.path.exists(dst):
        cv2.imwrite(dst, cv2.resize(crop_retina(cv2.imread(src)), (256, 256), interpolation=cv2.INTER_AREA))

def eval_all(df, tag, which=runs):
    global ext_loader
    ext_loader = DataLoader(ExtDS(df), batch_size=32, shuffle=False, num_workers=2)
    y, rows = df.diagnosis.values, []
    for mode, RUNN in which:
        ck = torch.load(f"{BASE_PATH}/checkpoints/best_{RUNN}.pth", map_location=device, weights_only=False)
        m = HybridMSLATNet(n_trans_layers=4, use_pos_emb=False, quantum_mode=mode, fixed_alpha=None).to(device)
        m.load_state_dict(ck["model"]); s, _ = summarize(y, predict(m)); del m
        rows.append(dict(run=RUNN, family=mode, **s)); print(tag, RUNN, {k: round(float(v), 4) for k, v in s.items()})
    return pd.DataFrame(rows)

# IDRiD with cropped borders
CACHE2 = "/content/idrid_crop256"; os.makedirs(CACHE2, exist_ok=True)
ext["cache"] = [f"{CACHE2}/{s}_{n}.png" for s, n in zip(ext.subset, ext.name)]
with ThreadPoolExecutor(4) as ex: list(tqdm(ex.map(cache_crop, zip(ext.path, ext.cache)), total=len(ext)))
r1 = eval_all(ext, "IDRiD-crop")
print(r1.groupby("family")[["acc", "f1", "qwk", "auc", "sens", "spec"]].agg(["mean", "std"]).round(3).to_string())

# control: the same crop on the APTOS test images (circuit seed 42 only)
os.makedirs("/content/aptos_crop", exist_ok=True)
a["cache"] = [f"/content/aptos_crop/{i}.png" for i in a.id_code]
for s_, d_ in zip(a.path, a.cache): cache_crop((s_, d_))
eval_all(a, "APTOS-crop", [runs[0]])      # known uncropped result: acc 0.8407, qwk 0.8759

100%|██████████| 516/516 [01:26<00:00,  5.96it/s]


IDRiD-crop v3_circuit_aNone_pe0_L4_s42 {'acc': 0.4709, 'f1': 0.3978, 'qwk': 0.6736, 'auc': 0.7596, 'sens': 0.7833, 'spec': 0.9119}


IDRiD-crop v3_circuit_aNone_pe0_L4_s43 {'acc': 0.4981, 'f1': 0.437, 'qwk': 0.7101, 'auc': 0.7596, 'sens': 0.8173, 'spec': 0.9223}


IDRiD-crop v3_circuit_aNone_pe0_L4_s44 {'acc': 0.5058, 'f1': 0.4038, 'qwk': 0.6553, 'auc': 0.8206, 'sens': 0.935, 'spec': 0.7098}


IDRiD-crop skip50_skip_aNone_pe0_L4_s42 {'acc': 0.5155, 'f1': 0.4269, 'qwk': 0.6947, 'auc': 0.7977, 'sens': 0.8762, 'spec': 0.8705}


IDRiD-crop skip50_skip_aNone_pe0_L4_s43 {'acc': 0.5136, 'f1': 0.4402, 'qwk': 0.6819, 'auc': 0.7979, 'sens': 0.8638, 'spec': 0.8653}


IDRiD-crop skip50_skip_aNone_pe0_L4_s44 {'acc': 0.5291, 'f1': 0.4274, 'qwk': 0.6922, 'auc': 0.7793, 'sens': 0.9009, 'spec': 0.7824}
           acc            f1          qwk           auc          sens          spec       
          mean    std   mean    std  mean    std   mean    std   mean    std   mean    std
family                                                                                    
circuit  0.492  0.018  0.413  0.021  0.68  0.028  0.780  0.035  0.845  0.080  0.848  0.120
skip     0.519  0.008  0.432  0.008  0.69  0.007  0.792  0.011  0.880  0.019  0.839  0.049


APTOS-crop v3_circuit_aNone_pe0_L4_s42 {'acc': 0.8348, 'f1': 0.597, 'qwk': 0.87, 'auc': 0.9255, 'sens': 0.9211, 'spec': 0.9244}


,run,family,acc,f1,qwk,auc,sens,spec
0,v3_circuit_aNone_pe0_L4_s42,circuit,0.834808,0.596964,0.870021,0.925465,0.921053,0.924444


In [ ]:
import glob
def stats(files):
    rgb, dark = [], []
    for f in files:
        im = cv2.imread(f)
        rgb.append(cv2.cvtColor(im, cv2.COLOR_BGR2RGB).reshape(-1, 3).mean(0))
        dark.append((cv2.cvtColor(im, cv2.COLOR_BGR2GRAY) < 12).mean())
    rgb = np.array(rgb)
    return "mean RGB", rgb.mean(0).round(1), "SD across images", rgb.std(0).round(1), "black fraction", round(float(np.mean(dark)), 3)
for nm, pat in [("APTOS test", "/content/aptos_chk/*.png"), ("IDRiD (uncropped)", "/content/idrid_256/*.png")]:
    fl = glob.glob(pat); print(nm, len(fl), *stats(fl))

APTOS test 339 mean RGB [106.4  56.9  18.8] SD across images [31.7 15.6 14.9] black fraction 0.237
IDRiD (uncropped) 516 mean RGB [111.   53.7  17.4] SD across images [19.1 12.9 12.8] black fraction 0.306


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, cohen_kappa_score
L = [0, 1, 2, 3, 4]
f = lambda P: np.log(np.clip(P, 1e-6, 1))
def m3(y, p): return (accuracy_score(y, p), f1_score(y, p, average="macro", labels=L, zero_division=0),
                      cohen_kappa_score(y, p, weights="quadratic", labels=L))
rows = []
for fam, pre in [("circuit", "v3_circuit_aNone_pe0_L4"), ("skip", "skip50_skip_aNone_pe0_L4")]:
    for s in (42, 43, 44):
        d = np.load(f"{BASE_PATH}/ext_idrid_{pre}_s{s}.npz"); P, Y = d["P"], d["Y"]
        tr, te = slice(0, 413), slice(413, 516)                    # IDRiD's own training / testing halves
        clf = LogisticRegression(max_iter=3000).fit(f(P[tr]), Y[tr])
        a0 = m3(Y[te], P[te].argmax(1)); a1 = m3(Y[te], clf.predict(f(P[te])))
        rows.append([fam, s, *np.round(a0, 3), *np.round(a1, 3)])
out = pd.DataFrame(rows, columns=["family", "seed", "acc0", "f1_0", "qwk0", "acc_cal", "f1_cal", "qwk_cal"])
print(out.to_string(index=False)); print(out.groupby("family").mean(numeric_only=True).round(3).drop(columns="seed"))

 family  seed  acc0  f1_0  qwk0  acc_cal  f1_cal  qwk_cal
circuit    42 0.359 0.333 0.533    0.515   0.404    0.598
circuit    43 0.379 0.344 0.534    0.495   0.367    0.595
circuit    44 0.359 0.297 0.258    0.456   0.324    0.456
   skip    42 0.379 0.322 0.454    0.485   0.394    0.492
   skip    43 0.388 0.339 0.470    0.573   0.458    0.577
   skip    44 0.330 0.268 0.453    0.485   0.374    0.531
          acc0   f1_0   qwk0  acc_cal  f1_cal  qwk_cal
family                                                
circuit  0.366  0.325  0.442    0.489   0.365    0.550
skip     0.366  0.310  0.459    0.514   0.409    0.533
